In [ ]:
"""
IMPORT ALL IMPORTS, DEPENDENCIES, ETC NEEDED
"""
from huggingface_hub import login
from datasets import load_dataset
import unicodedata
import pandas as pd
import torch
from transformers import (AutoTokenizer,AutoModelForSeq2SeqLM,Trainer,TrainingArguments,)
from torch.utils.data import DataLoader
import editdistance
import torch.nn as nn

In [ ]:
#INSERT YOUR HUGGINGFACE READ-ONLY TOKEN HERE INSIDE QUOTES
login("INSERT TOKEN HERE")

#LOAD THE DATASET, EXTRACT ONLY THE TEXT ENTRIES
ds = load_dataset("acflp/YANKARI", split="train")
ds_text = ds.remove_columns([col for col in ds.column_names if col != "text"])

"""
INITIAL DATA PREPROCESSING

THIS ENDS WITH PREPPED TRAINING DATASET SPLIT
PREPPED TRAINING ENTRIES INCLUDE:
1. Input text with no diacritic markings
2. Matching target text with diacritic markings

FORMAT:
{
    "input_text": "...",
    "target_text": "..."
}
"""

#RANDOMLY SPLIT 50% TRAIN
split_1 = ds_text.train_test_split(test_size=0.5, seed=42)
train_ds = split_1['train']
temp_ds = split_1['test']
#RANDOMLY SPLIT 25% DEV, 25% TEST - Not used in this run
split_2 = temp_ds.train_test_split(test_size=0.5, seed=42)
dev_ds = split_2['train']
test_ds = split_2['test']

In [ ]:
#DIACRITIC STRIPPING FUNCTION
#FOR CREATING DIACRITIC-FREE INPUT ENTRIES
def strip_diacritics(text):
    return ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
    )

"""
DIACRITIC LABEL FUNCTION
0 = none
1 = dot only
2 = high only
3 = low only
4 = dot + high
5 = dot + low
"""
def get_byte_diacritic_labels(text, max_len=256):
    #INSTANTIATE LABEL LIST
    labels = []
    #FOR ALL TEXTS
    for c in text:
        #DECOMPOSE TO UNIVODE
        decomposed = unicodedata.normalize('NFD', c)
        marks = [ch for ch in decomposed if unicodedata.category(ch) == 'Mn']
        #SET DIACRITIC CONDITIONS TO FALSE
        has_dot = False
        has_high = False
        has_low = False
        #LOGIC FOR DIACRITICS
        for m in marks:
            if m == '\u0323':
                has_dot = True
            elif m == '\u0301':
                has_high = True
            elif m == '\u0300':
                has_low = True
        #ASSIGN LABELS FOR ALL POSSIBLE YORUBA DIACRITIC CLASSES
        if has_dot and has_high:
            label = 4
        elif has_dot and has_low:
            label = 5
        elif has_dot:
            label = 1
        elif has_high:
            label = 2
        elif has_low:
            label = 3
        else:
            label = 0

        byte_len = len(c.encode("utf-8"))
        #APPEND TO LABEL LIST
        labels.append(label)
        labels.extend([-100] * (byte_len - 1))

    labels = labels[:max_len]
    labels += [-100] * (max_len - len(labels))

    return labels

#FUNCTION TO PREP INPUT
#GIVEN ENTRY, MAKES DIACRITIC-FREE INPUT, DIACRITIC MARKED TARGET
def preprocess_hf(example):
    return {
        "input_text": strip_diacritics(example["text"]),
        "target_text": example["text"]
    }

In [ ]:
#RUN TEXT PREPROCESSING ON ALL SPLITS
train_ds = train_ds.map(preprocess_hf, remove_columns=['text'])
dev_ds = dev_ds.map(preprocess_hf, remove_columns=['text'])
test_ds = test_ds.map(preprocess_hf, remove_columns=['text'])
#USE TRAIN SET FOR TRAINING
dataset = train_ds

In [ ]:
"""
MODEL SETUP

THIS ENDS WITH BYT5 MODEL LOADED, TOKENIZATION FUNCTIONS READY
"""

#LOAD BYT5 MODEL AND TOKENIZER
#CURRENTLY USING SMALL, WE CAN TRY UPPING TO BYT5 BASE
model_name = "google/byt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

#FREEZE ENCODER LAYERS
#CURRENTLY FREEZES HALF TO REDUCE TRAINING TIME, WE CAN TRY UNFREEZING MORE
num_encoder_layers = len(model.encoder.block)
for layer in model.encoder.block[:num_encoder_layers // 2]:
    for param in layer.parameters():
        param.requires_grad = False

In [ ]:
#INSTANTIATE LINEAR DIACRITIC HEAD
class DiacriticHead(nn.Module):
    def __init__(self, hidden_size, num_classes=6):
        super().__init__()
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, hidden_states):
        return self.classifier(hidden_states)

In [ ]:
model.diacritic_head = DiacriticHead(model.config.d_model)

#TOKENIZATION FUNCTION
MAX_LEN = 256

In [ ]:
def preprocess(example):
    #TOKENIZE INPUT
    model_inputs = tokenizer(
        example["input_text"],
        truncation=True,
        #PADS SHORT ENTRIES TO 256
        padding="max_length",
        #CAPS LONG ENTRIES TO 256
        #WE MIGHT WANT TO EXTEND THIS, OR IMPLEMENT SLIDING WINDOW, SINCE THIS LOSES INPUT INFORMATION
        max_length=MAX_LEN,
    )
    #TOKENIZE TARGET, SAME SETUP
    labels = tokenizer(
        example["target_text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )["input_ids"]

    #PADDING TOKENS
    #LIST COMPREHENSION FOR ALL LABEL IDS, WITH ALL PADDING TOKENS SET TO -100
    #CROSS ENTROPY LOSS IGNORES VALUE -100
    labels = [(l if l != tokenizer.pad_token_id else -100) for l in labels]

    diacritic_labels = get_byte_diacritic_labels(example["target_text"], MAX_LEN)

    model_inputs["labels"] = labels
    model_inputs["diacritic_labels"] = diacritic_labels
    return model_inputs

In [ ]:
"""
RUN TOKENIZATION OF TRAINING SPLIT

ENDS WITH TOKENIZED TRAINING SPLIT
"""

#GET TOKENIZED DATASET
tokenized_dataset = dataset.map(preprocess)

#FORMAT TOGETHER WITH INPUTS, ATTENTION MASK, LABELS
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels", "diacritic_labels"]
)

In [ ]:
"""
DEFINE TRAINER CLASS, SET TRAINING HYPERPARAMETERS

ENDS WITH MODEL READY TO TRAIN
"""
#TRAINER CLASS
class YorubaTrainer(Trainer):
    def __init__(self, alpha, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.alpha = alpha

        #WEIGHTS (6 classes)
        self.diacritic_loss_fn = nn.CrossEntropyLoss(
            weight=torch.tensor([1.0, 5.0, 5.0, 5.0, 6.0, 6.0])
        )
    #FUNCTION TO GET LOSS
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]
        diacritic_labels = inputs["diacritic_labels"]

        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            labels=labels,
            output_hidden_states=True
        )

        main_loss = outputs.loss
        #FROM LAST HIDDEN REPRESENTATION, SEND ENCODINGS TO DIACRITIC HEAD
        decoder_hidden = outputs.decoder_hidden_states[-1]
        diacritic_logits = model.diacritic_head(decoder_hidden)

        self.diacritic_loss_fn.weight = self.diacritic_loss_fn.weight.to(diacritic_logits.device)

        mask = (labels != -100) & (diacritic_labels != -100)
        #GET DIACRITIC SPECIFIC LOSS
        diacritic_loss = self.diacritic_loss_fn(
            diacritic_logits.view(-1, 6)[mask.view(-1)],
            diacritic_labels.view(-1)[mask.view(-1)]
        )
        diacritic_loss = diacritic_loss / mask.sum()
        #MODULATE MAIN LOSS WITH alpha * DIACRITIC LOSS
        loss = main_loss + self.alpha * diacritic_loss

        return (loss, outputs) if return_outputs else loss


In [ ]:
#TRAINING VARIABLES
training_args = TrainingArguments(
    #SAVES MODEL PARAMETERS WHEN DONE
    output_dir="../diacritic_head_alpha01",
    learning_rate=5e-4,
    #TRAIN BATCH SIZE, MAKE SURE SAME AS EVAL
    per_device_train_batch_size=16,
    num_train_epochs=15,
    weight_decay=0.01,
    #FOR PROGRESS MONITORING - PRINTS LOSS EVERY N STEPS DURING TRAINING
    logging_steps=50,
    #CURRENTLY NO INTERMITENT SAVING, WAS BREAKING
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False
)

#ACTUAL TRAINER
trainer = YorubaTrainer(
    #PASSES IN ALL OF ABOVE
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    alpha=0.01
)

In [ ]:
"""
RUN MODEL TRAINING
"""

#RUN THE TRAINER
trainer.train()

#CHANGE PATHS INSIDE QUOTES TO CHANGE SAVE DIRECTORY
model.save_pretrained("./diacritic_head_alpha01")
tokenizer.save_pretrained("/diacritic_head_alpha01")